# WR populations

I would like to find out if there are more or less interactions as metallicity decreases, because this could affect the origin of the decrease in WR numbers.

To this end, I'm loading in a complete population at a single metallicity and determine what systems are interacting and which are not.

$
\log_{10}(L_\mathrm{threshold}/L_\odot) = 4.9 - \log(Z/Z_\odot)
$

In [ ]:
def wr_duration_partial(dataframe, star='S1', H_surface_threshold=0.4, get_index=False, return_interp=False, accept_direct_collapse=False):
    """
    Find the final (latest-in-time) partial-WR region where surface_he4 > He_surface_threshold.
    By default the region is accepted only if a SN row follows the star's CC. If
    accept_direct_collapse=True then a CC without an explicit SN (direct collapse) is
    also accepted.
    Returns same shape/signature as other wr_duration_* helpers.
    """

    import numpy as np
    import pandas as pd

    try:
        threshold = float(H_surface_threshold)
    except Exception:
        raise ValueError("H_surface_threshold must be convertible to float")
    if not (0.0 <= threshold < 1.0):
        raise ValueError('H_surface_threshold must be between 0 and 1') 
    df = dataframe.copy()
    # dfr has a positional index and preserves original labels in 'orig_index'
    # dfr = df.reset_index().rename(columns={'index': 'orig_index'}).reset_index(drop=True)
    # print('orig_index values:', dfr['orig_index'].tolist())

    df['orig_index'] = df.index.to_numpy()
    dfr = df[['orig_index'] + list(df.columns.drop('orig_index'))].reset_index(drop=True)

    # dfr = df.reset_index()
    # dfr['orig_index'] = df.index.to_numpy()

    needed_columns = [f'{star}_surface_h1', 'time', 'event', 'step_names']

    surface = dfr[f'{star}_surface_h1'].astype(float).fillna(0.0)
    times = dfr['time'].astype(float).values
    cc_exists_mask = dfr['event'].fillna('').str.contains('CC', na=False)

    # helper: operate on dfr (positional) to keep consistent mapping between start/end
    def find_end_after_label(start_label):
        cc_pat = 'CC1' if star == 'S1' else 'CC2'
        # select rows at or after the provided original-label start_label
        df_after = dfr[dfr['orig_index'] >= start_label].reset_index(drop=True)
        cc_mask = df_after['event'].fillna('').str.contains(cc_pat, na=False)
        if not cc_mask.any():
            return None, None
        # cc_orig_label is the original df label of first CC after start_label
        cc_row_pos = int(np.flatnonzero(cc_mask)[0])
        cc_orig_label = int(df_after.loc[cc_row_pos, 'orig_index'])
        # prefer step_SN row that follows the CC (same CC) within dfr
        # find step_SN rows where previous event == cc_pat and orig_index >= cc_orig_label
        df_sn_candidates = dfr[dfr['orig_index'] >= cc_orig_label].reset_index(drop=True)
        # condition: step_names == 'step_SN' AND event.shift(1) == cc_pat (use df_sn_candidates)
        # to compute shifted event relative to original dfr we need to use positions in dfr:
        # find positions in original dfr where step_names == 'step_SN' and the previous row's event == cc_pat
        step_sn_mask = (dfr['step_names'].fillna('') == 'step_SN')
        prev_event = dfr['event'].shift(1).fillna('')
        sn_assoc_mask = step_sn_mask & (prev_event == cc_pat) & (dfr['orig_index'] >= cc_orig_label)

        print('no. of True sn_assoc_mask:', sn_assoc_mask.sum())
        
        if sn_assoc_mask.any():
            sn_label = int(dfr.loc[sn_assoc_mask, 'orig_index'].min())
            sn_time = float(dfr.loc[dfr['orig_index'] == sn_label, 'time'].iloc[0])
            return sn_label, sn_time
        # fallback: any SN event after CC (look for event containing 'SN')
        df_after_cc = dfr[dfr['orig_index'] >= cc_orig_label].reset_index(drop=True)

        print('no. of rows in df_after_cc:', df_after_cc.shape[0]) #should be 1

        sn_mask = df_after_cc['event'].fillna('').str.contains(r'\bSN\b', na=False)

        print('no. of True sn_mask:', sn_mask.sum()) #should be 0 or 1
        if sn_mask.any():
            sn_row_pos = int(np.flatnonzero(sn_mask)[0])
            sn_label = int(df_after_cc.loc[sn_row_pos, 'orig_index'])
            sn_time = float(dfr.loc[dfr['orig_index'] == sn_label, 'time'].iloc[0])
            return sn_label, sn_time
        # no SN after CC -> either accept CC as direct collapse or mark as no valid end
        if accept_direct_collapse:
            return int(cc_orig_label), float(dfr.loc[dfr['orig_index'] == cc_orig_label, 'time'].iloc[0])
        return None, None
    
    # find contiguous regions where surface < threshold
    mask = (surface < threshold).to_numpy()
    idxs = np.flatnonzero(mask)
    groups = np.split(idxs, np.where(np.diff(idxs) != 1)[0] + 1)
    
    print('groups are', groups)
    #print('group start and ends are', [(int(groups[i][0]), int(groups[i][-1])) for i in range(len(groups))])

    if not groups:
        print('no groups found for star', star)
        return None, [], None, []

    # valid_regions = []
    if len(groups) > 1:
        final_group = groups[-1] #picks the last one
    else:
        final_group = groups[0] #check this
    #below 4 lines were at this indentation
    print('final group is', final_group)
    if len(final_group) == 0:
        # no WR region found
        return None, [], None, [] 
    
    partial_wr_start_pos = final_group[0] #starting index
    #find indices:"
    before_start_index = partial_wr_start_pos - 1
    surface_before = float(surface.iloc[before_start_index])
    surface_at_start = float(surface.iloc[partial_wr_start_pos]) 
    time_before_start = float(times[before_start_index])
    time_at_start = float(times[partial_wr_start_pos])
    frac = (threshold - surface_before) / (surface_at_start - surface_before)
    frac = float(np.clip(frac, 0.0, 1.0))
    interpolated_start_time = time_before_start + frac * (time_at_start - time_before_start)

    end_index = final_group[-1]
    if end_index + 1 >= len(dfr):
        # no row after end_index to mark end of WR
        print('end_index + 1 is > len(dfr)')
        return None, [], None, []
    # find the end time at the row after end_index
    end_time = float(dfr['time'].iloc[end_index + 1]) #at the end of WR. For ex. this might be the SN entry.
    #verifying steps:
    print('end_index is', end_index, 'at end time of', end_time)
    print(dfr.loc[end_index - 5: end_index + 5,[f'{star}_state', 'step_names', 'event']].fillna(''))

    WR_index_range = [partial_wr_start_pos, end_index]   # original df labels (consistent)
    WR_time_list = [interpolated_start_time, end_time]
    WR_interp_start = interpolated_start_time

    WR_duration = WR_time_list[1] - WR_time_list[0] if WR_time_list[0] is not None and WR_time_list[1] is not None else None

    if get_index:
        print(f"{star}: partial WR for {WR_duration} d, spanning {WR_time_list} d, interp start={WR_interp_start}, WR_index_range={WR_index_range}")
        if return_interp:
            return WR_duration, WR_time_list, WR_index_range, WR_interp_start
        return WR_duration, WR_time_list, WR_index_range
    else:
        print(f"{star}: partial WR for {WR_duration} d, spanning {WR_time_list} d, interp start={WR_interp_start}")
        if return_interp:
            return WR_duration, WR_time_list, WR_interp_start
        return WR_duration, WR_time_list
#%%

def partial_stripped_selection_function(history_chunk, oneline_chunk, formation_channels_chunk):
    '''A selection function to create a transient population of stripped stars.
    Recall: A transient population is only one row per binary system. So we want properties that last one entry.
    Per binary, to include:
    Oneline headers:
    - 'wr_start' using start time of surface_he4 >= 0.6. Use the functions w interpolation!
    - 'wr_end'
    - 'wr_duration'

    Make sure the chunks consist of one binary system for each row!
    311025-1243h: Test loop with twenty_histories and twenty_onelines first.
    '''
    surface_H1_threshold = 0.4
    indices = oneline_chunk.index.to_numpy()

    #args in bf.calculate_wr_duration_partial: df, 'S1' or 'S2', surface_he4_threshold, get_index=False, return_interp=False, accept_direct_collapse=False
    #set get_index to True and remember to do for both.
    #You need to call threshold again as still, only certain times have surface_he4 >= 0.6

    df_transients = pd.DataFrame(index = indices) #this creates the "base"
    #len_hists = len(history_chunk.index.unique()) #20

    for binary_idx in indices:
      # get the per-binary history and make a safe copy
        indiv_hist_initial = history_chunk.loc[binary_idx].copy()

        # if .loc returned a Series (single row), convert to a 1-row DataFrame
        
        if isinstance(indiv_hist_initial, pd.Series):
            indiv_hist_initial = indiv_hist_initial.to_frame().T.reset_index(drop=True)

        # add orig_index column containing the original index values (preserves tuples for MultiIndex)
        indiv_hist_initial['orig_index'] = indiv_hist_initial.index.to_numpy()

        # put orig_index first and reset positional index so iloc works reliably
        cols = ['orig_index'] + [c for c in indiv_hist_initial.columns if c != 'orig_index']
        indiv_hist = indiv_hist_initial[cols].reset_index(drop=True)

        indiv_oneline = oneline_chunk.loc[binary_idx] #df
        print('BINARY IDX', binary_idx)
        #note: these are indices within each indiv_hist (i.e. a binary), not the overall df
        S1_wr_duration, S1_wr_time_list, S1_index_range, S1_interp_start = wr_duration_partial(indiv_hist, 'S1', surface_H1_threshold, get_index=True, return_interp=True, accept_direct_collapse=True)
        S2_wr_duration, S2_wr_time_list, S2_index_range, S2_interp_start = wr_duration_partial(indiv_hist, 'S2', surface_H1_threshold, get_index=True, return_interp=True, accept_direct_collapse=True)        
        if S1_wr_duration is not None:
            
            S1_wr_index_start = S1_index_range[0]
            S1_wr_index_end = S1_index_range[1]
            print('len hist', len(indiv_hist), 'S1_index_range', S1_index_range)
            assert 0 <= S1_wr_index_start < len(indiv_hist), "start out of bounds"
            assert 0 <= S1_wr_index_end <= len(indiv_hist), "end out of bounds (end may be exclusive)"


            df_transients.loc[binary_idx, 'S1_wr_start'] = S1_wr_time_list[0]
            df_transients.loc[binary_idx, 'S1_wr_start_index'] = S1_wr_index_start
            df_transients.loc[binary_idx, 'S1_wr_end'] = S1_wr_time_list[1]
            df_transients.loc[binary_idx, 'S1_wr_end_index'] = S1_wr_index_end
            # df_transients.loc[binary_idx, 'S1_wr_mean'] = np.mean(S1_wr_time_list)
            df_transients.loc[binary_idx, 'S1_wr_duration'] = S1_wr_duration
            df_transients.loc[binary_idx, 'S1_log_L_wr_start'] = indiv_hist.iloc[S1_wr_index_start]['S1_log_L']
            df_transients.loc[binary_idx, 'S1_log_L_wr_end'] = indiv_hist.iloc[S1_wr_index_end]['S1_log_L']
            df_transients.loc[binary_idx, 'S1_mass_wr_start'] = indiv_hist.iloc[S1_wr_index_start]['S1_mass']
            df_transients.loc[binary_idx, 'S1_mass_wr_end'] = indiv_hist.iloc[S1_wr_index_end]['S1_mass']
    
            # also store the interpolation starts returned by wr_duration_partial
            df_transients.loc[binary_idx, 'S1_interp_start'] = S1_interp_start

        df_transients.loc[binary_idx, 'S1_mass_i'] = indiv_oneline['S1_mass_i']
        if S1_wr_duration is None:
            print('No S1 WR phase for binary index', binary_idx)
        
        if S2_wr_duration is not None:
            # placeholder columns to fill later per-row (recommend per-row assignment instead of whole-column writes)
            # df_transients['S1_log_L']
            # df_transients['S1_mass_list']
            # df_transients['S1_surface_h1_list']
            # df_transients['S1_surface_he4_list'] = 
            #TEST WITHOUT COMPLEXITIES OF ADDINNG LISTS FIRST.
            S2_wr_index_start = S2_index_range[0] #old indices
            S2_wr_index_end = S2_index_range[1]
            # similarly for S2 if needed
            assert 0 <= S2_wr_index_start < len(indiv_hist), "start out of bounds"
            assert 0 <= S2_wr_index_end <= len(indiv_hist), "end out of bounds (end may be exclusive)"
            
            df_transients.loc[binary_idx, 'S2_wr_start'] = S2_wr_time_list[0]
            df_transients.loc[binary_idx, 'S2_wr_end'] = S2_wr_time_list[1]
            df_transients.loc[binary_idx, 'S2_wr_start_index'] = S2_wr_index_start
            df_transients.loc[binary_idx, 'S2_wr_end_index'] = S2_wr_index_end
            # df_transients.loc[binary_idx, 'S2_wr_mean'] = np.mean(S2_wr_time_list)
            df_transients.loc[binary_idx, 'S2_wr_duration'] = S2_wr_duration

            df_transients.loc[binary_idx, 'S2_log_L_wr_start'] = indiv_hist.iloc[S2_wr_index_start]['S2_log_L']
            df_transients.loc[binary_idx, 'S2_log_L_wr_end'] = indiv_hist.iloc[S2_wr_index_end]['S2_log_L']
            df_transients.loc[binary_idx, 'S2_mass_wr_start'] = indiv_hist.iloc[S2_wr_index_start]['S2_mass']
            df_transients.loc[binary_idx, 'S2_mass_wr_end'] = indiv_hist.iloc[S2_wr_index_end]['S2_mass']
        
            df_transients.loc[binary_idx, 'S2_interp_start'] = S2_interp_start
            
        df_transients.loc[binary_idx, 'S2_mass_i'] = indiv_oneline['S2_mass_i']
        if S2_wr_duration is None:
            print('No S2 WR phase for binary index', binary_idx)
        
        
        
    df_transients['time'] = 1 #force an arbitrary time column or else it will fail

    df_transients['metallicity'] = oneline_chunk['metallicity']

    #i didn't include interp_start as it is the same as {star}_wr_start
    #test for tau etc later. But you don't have log_mdot activated

    # Added properties. This is where you define what properties you wish to extract.
    
    #mask = (history_chunk['S1_state'] == 'BH') & (history_chunk['S2_state'] == 'BH') & (history_chunk['step_names'] == 'step_SN') & (history_chunk['state'] == 'detached')
    #use np.isany() to see if S1 or S2 reaches surface_He4 >= 0.6
    return df_transients

#%% NOTE: Before the above functions are done, he previous steps have to be done:
# Out of the 100000 binaries, select the binary systems with WR times (total 51406 binary_indexes)
import pandas as pd
level = 'binary_index' if 'binary_index' in tmp_data.index.names else None

mask = (tmp_data['S1_surface_he4'] >= 0.6) | (tmp_data['S2_surface_he4'] >= 0.6)
hits = tmp_data[mask].index
ids = hits.get_level_values(level) if level is not None else hits
unique_ids = pd.Index(ids).unique().astype(int).tolist()

print("Number of binaries with WR:", len(unique_ids))
print("Examples:", unique_ids[:10])
#%%
# Load population 
pop = Population(f'../SN_workshop/binaries_with_WR.h5', chunksize=1000)

#%%


    
    

In [ ]:
from posydon.utils.common_functions import convert_metallicity_to_string
import numpy as np
import pandas as pd


data_folder = '/home/users/b/briel/scratch/WR_populations/population_synthesis/runs/2025_11_04/'
metallicity = 1.0  # in units of Z_sun
met_str = convert_metallicity_to_string(metallicity)
metallicity_file = data_folder + f"{met_str}_Zsun_population.h5"

: 

In [ ]:
from posydon.popsyn.synthetic_population import Population
pop = Population(metallicity_file)

: 

: 

In [ ]:
tmp_data = pop.oneline[['interp_class_HMS_HMS', 'interp_class_CO_HMS_RLO', 'mt_history_HMS_HMS', 'mt_history_CO_HMS_RLO']]

: 

In [ ]:
value_counts = tmp_data['interp_class_HMS_HMS'].value_counts()

unstable_mt_count = value_counts['unstable_MT'] 
stable_mt_count = value_counts['stable_MT']
no_mt_count = value_counts['no_MT']
stable_reverse_mt_count = value_counts['stable_reverse_MT']

interacting_count = unstable_mt_count + stable_mt_count + stable_reverse_mt_count

interacting_fraction = interacting_count / tmp_data.shape[0]

print(f"Total systems: {tmp_data.shape[0]}"
      f"\nInteracting systems: {interacting_count}"
      f"\nInteracting fraction: {interacting_fraction*100:.2f}%")


: 

In [ ]:
# S1 only
tmp_hist = pop.history[['S1_surface_h1', 'time', 'event', 'step_names', 'S1_log_L', 'S1_mass']]

: 

In [ ]:
import plotly.graph_objects as go

In [ ]:
value_counts = tmp_data['interp_class_HMS_HMS'].value_counts()

unstable_mt_count = value_counts.get('unstable_MT', 0)
stable_mt_count = value_counts.get('stable_MT', 0)
no_mt_count = value_counts.get('no_MT', 0)
stable_reverse_mt_count = value_counts.get('stable_reverse_MT', 0)

total_systems = tmp_data.shape[0]

# Define nodes
# Node 0: All Systems (left)
# Node 1: Unstable MT (right, top)
# Node 2: Stable MT (right, middle-top)
# Node 3: Stable Reverse MT (right, middle-bottom)
# Node 4: No MT (bottom)

node_labels = [
    f"All Systems<br>({total_systems:,})",
    f"No MT<br>({no_mt_count:,})",
    f"Interacting Systems<br>({unstable_mt_count + stable_mt_count + stable_reverse_mt_count:,})",
    f"Stable MT<br>({stable_mt_count:,})",
    f"Unstable MT<br>({unstable_mt_count:,})",
    f"Stable Reverse MT<br>({stable_reverse_mt_count:,})",
    
]

# Define links
source = [0, 0, 2, 2, 2]  
target = [1, 2, 3, 4, 5] 
value = [no_mt_count, 
         stable_mt_count+unstable_mt_count+stable_reverse_mt_count,
         stable_mt_count,
         unstable_mt_count,
         stable_reverse_mt_count]
#value = [unstable_mt_count, stable_mt_count, stable_reverse_mt_count, no_mt_count]

# Custom node positions
# x: 0 for left (all systems), 1 for right (interacting types) and bottom (no MT)
# y: vertical position
node_x = [0.1, 0.3, 0.3, 0.7, 0.7, 0.7]
node_y = [0.5, 0.92, 0.45, 0.1, 0.52, 0.75]


fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
    )
)])

fig.update_layout(
    title=f"Binary System Interactions (Z = {metallicity}"+r"$Z_\odot$)",
    font=dict(size=12),
    height=600,
    width=900
)

fig.show()

## across metallicity

In [36]:
from posydon.utils.common_functions import convert_metallicity_to_string
import numpy as np
import pandas as pd

metallicities = [0.01, 0.1, 0.2, 0.45, 1.0]
for met in metallicities:
    data_folder = '/home/users/b/briel/scratch/WR_populations/population_synthesis/runs/2025_11_04/'
    metallicity = met  # in units of Z_sun
    print(f"Metallicity: {metallicity} Z_sun")
    met_str = convert_metallicity_to_string(metallicity)

    metallicity_file = data_folder + f"{met_str}_Zsun_population.h5"

    pop = Population(metallicity_file)

    tmp_data = pop.oneline[['interp_class_HMS_HMS', 'interp_class_CO_HMS_RLO', 'mt_history_HMS_HMS', 'mt_history_CO_HMS_RLO']]
    value_counts = tmp_data['interp_class_HMS_HMS'].value_counts()

    unstable_mt_count = value_counts['unstable_MT'] 
    stable_mt_count = value_counts['stable_MT']
    no_mt_count = value_counts['no_MT']
    stable_reverse_mt_count = value_counts['stable_reverse_MT']

    interacting_count = unstable_mt_count + stable_mt_count + stable_reverse_mt_count

    interacting_fraction = interacting_count / tmp_data.shape[0]

    print(f"Total systems: {tmp_data.shape[0]}"
        f"\nInteracting systems: {interacting_count}"
        f"\nInteracting fraction: {interacting_fraction*100:.2f}%"
        f"\nStable MT: {stable_mt_count} ({(stable_mt_count/tmp_data.shape[0])*100:.2f}%)"
        f"\nUnstable MT: {unstable_mt_count} ({(unstable_mt_count/tmp_data.shape[0])*100:.2f}%)"
        f"\nStable Reverse MT: {stable_reverse_mt_count} ({(stable_reverse_mt_count/tmp_data.shape[0])*100:.2f}%)"
    )


Metallicity: 0.01 Z_sun
Total systems: 100000
Interacting systems: 89157
Interacting fraction: 89.16%
Stable MT: 45718 (45.72%)
Unstable MT: 42406 (42.41%)
Stable Reverse MT: 1033 (1.03%)
Metallicity: 0.1 Z_sun
Total systems: 100000
Interacting systems: 89391
Interacting fraction: 89.39%
Stable MT: 41792 (41.79%)
Unstable MT: 46564 (46.56%)
Stable Reverse MT: 1035 (1.03%)
Metallicity: 0.2 Z_sun
Total systems: 100000
Interacting systems: 89794
Interacting fraction: 89.79%
Stable MT: 42082 (42.08%)
Unstable MT: 46537 (46.54%)
Stable Reverse MT: 1175 (1.18%)
Metallicity: 0.45 Z_sun
Total systems: 100000
Interacting systems: 89240
Interacting fraction: 89.24%
Stable MT: 41315 (41.32%)
Unstable MT: 46727 (46.73%)
Stable Reverse MT: 1198 (1.20%)
Metallicity: 1.0 Z_sun
Total systems: 100000
Interacting systems: 87669
Interacting fraction: 87.67%
Stable MT: 41444 (41.44%)
Unstable MT: 45168 (45.17%)
Stable Reverse MT: 1057 (1.06%)


Looking at just the first interaction, the following conclusions can be drawn:


1. For the total interacting systems, there's only a small change in the % of interacting systems. Only at 1\Zsun is there a jump down in interacting systems. Most likely from strong winds stopping intactions.
2. The percentage of stable and unstbale systems remains 